# N1 · SciBERT Named Entity Recognition (SciERC)
NLP track, model 1. Fine-tunes `allenai/scibert_scivocab_uncased` to tag **Task, Method, Metric, Material,
OtherScientificTerm, Generic** in scientific text. Baseline: general-domain `bert-base-uncased`.
Every run saves `metrics.json`, `history.csv`, `predictions.jsonl` to Drive and resumes automatically after a disconnect.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN` in the Secrets panel (key icon) with notebook access on.
Edit the two settings below once.

In [ ]:
REPO_URL = "https://github.com/mkavinvelavan/researchlens.git"   # <- change
HF_USERNAME = "YOUR_HF_USERNAME"                                          # <- change
DRIVE_ROOT = "/content/drive/MyDrive/ResearchLens"                        # checkpoints + results live here

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists("/content/researchlens"):
    !git clone -q {REPO_URL} /content/researchlens
else:
    !git -C /content/researchlens pull -q
%cd /content/researchlens
!pip install -q -r requirements-colab.txt

os.environ["HF_USERNAME"] = HF_USERNAME
os.environ["RL_DRIVE_ROOT"] = DRIVE_ROOT
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Train SciBERT (3 seeds; seed 42 is pushed to the Hugging Face Hub)

In [ ]:
SEEDS = [42, 43, 44]
for s in SEEDS:
    push = "--push_to_hub" if s == 42 else ""
    !python -m researchlens.nlp.train_ner --config configs/n1_ner.yaml --seed {s} {push}

## 2. Baseline: general-domain BERT (same settings)

In [ ]:
for s in SEEDS:
    !python -m researchlens.nlp.train_ner --config configs/n1_ner.yaml --model_name bert-base-uncased --run_name bert_base --seed {s}

## 3. Report: all graphs + tables

In [ ]:
from researchlens.evaluation.reports import ner_report
REPORT_DIR = f"{DRIVE_ROOT}/results/n1_ner/report"
display(ner_report(f"{DRIVE_ROOT}/results"))

from pathlib import Path
from IPython.display import Image, display
for f in sorted(Path(REPORT_DIR).glob("*.png")):
    print(f.name); display(Image(str(f), width=650))